# Milestone #4 — Regression DiD + Two-way Fixed Effects DiD

Combined notebook for issues **#29** and **#31**. Uses only the US + Canada subset, reuses `src/simple_did.py`, saves both rows to `results/estimates.csv`, and creates a four-way comparison against the prior naive and manual 2×2 DiD estimates.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Works from notebooks/simple_did or notebooks/. Finds repo root by looking for src/simple_did.py.
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "src" / "simple_did.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find repo root containing src/simple_did.py")

sys.path.append(str(PROJECT_ROOT / "src"))
import simple_did as sd

TREATED = "United States"
CONTROL = "Canada"
METRIC = sd.METRIC  # revenue_per_session
AUTHOR = "@gaurvsingh095"

df = sd.load_panel()
us_ca = df[df["market"].isin([TREATED, CONTROL])].copy()

print("Project root:", PROJECT_ROOT)
print("Metric:", METRIC)
print("US + Canada rows:", len(us_ca))

Project root: .
Metric: revenue_per_session
US + Canada rows: 208


## Simple regression DiD (#29)

Model: `revenue_per_session ~ pilot_market + post_launch + pilot_market:post_launch`. The interaction is the estimated ATT.

In [2]:
formula = f"{METRIC} ~ pilot_market + post_launch + pilot_market:post_launch"
simple_model = sd.smf_ols(formula, us_ca)  # Newey-West/HAC errors, maxlags=8 by default
term = "pilot_market:post_launch"
ci = simple_model.conf_int().loc[term]

simple_att = float(simple_model.params[term])
simple_se = float(simple_model.bse[term])
simple_p = float(simple_model.pvalues[term])
simple_ci_low = float(ci[0])
simple_ci_high = float(ci[1])

simple_result = pd.DataFrame([{
    "model": "simple_regression_did",
    "term": term,
    "att": simple_att,
    "std_err": simple_se,
    "p_value": simple_p,
    "ci_low": simple_ci_low,
    "ci_high": simple_ci_high,
    "nobs": int(simple_model.nobs),
}])

simple_result

                   model                      term  ...   ci_high  nobs
0  simple_regression_did  pilot_market:post_launch  ...  0.512167   208

[1 rows x 8 columns]

In [3]:
sd.save_estimate(
    method="simple_regression_did",
    estimate=simple_att,
    issue=29,
    author=AUTHOR,
    treated=TREATED,
    control=CONTROL,
    metric=METRIC,
    window="all_pre_vs_post",
    ci_low=simple_ci_low,
    ci_high=simple_ci_high,
    notes="Simple regression DiD using US + Canada only. Coefficient on pilot_market:post_launch; OLS with Newey-West/HAC errors via sd.smf_ols(maxlags=8).",
)

print(f"Simple regression DiD ATT: {simple_att:+.4f} RPS")
print(f"95% CI: [{simple_ci_low:+.4f}, {simple_ci_high:+.4f}]")
print(f"p-value: {simple_p:.4f}")

Simple regression DiD ATT: +0.2459 RPS
95% CI: [-0.0203, +0.5122]
p-value: 0.0703


## Two-way fixed-effects DiD (#31)

Model: `revenue_per_session ~ digital_feature_available + C(market) + C(week_start)`. The `digital_feature_available` coefficient is the estimated ATT after absorbing market and week fixed effects.

In [4]:
formula = f"{METRIC} ~ digital_feature_available + C(market) + C(week_start)"
fe_model = sd.smf_ols(formula, us_ca)  # Newey-West/HAC errors, maxlags=8 by default
term = "digital_feature_available"
ci = fe_model.conf_int().loc[term]

fe_att = float(fe_model.params[term])
fe_se = float(fe_model.bse[term])
fe_p = float(fe_model.pvalues[term])
fe_ci_low = float(ci[0])
fe_ci_high = float(ci[1])

fe_result = pd.DataFrame([{
    "model": "two_way_fe_did",
    "term": term,
    "att": fe_att,
    "std_err": fe_se,
    "p_value": fe_p,
    "ci_low": fe_ci_low,
    "ci_high": fe_ci_high,
    "nobs": int(fe_model.nobs),
}])

fe_result

            model                       term  ...   ci_high  nobs
0  two_way_fe_did  digital_feature_available  ...  0.303897   208

[1 rows x 8 columns]

In [5]:
sd.save_estimate(
    method="two_way_fe_did",
    estimate=fe_att,
    issue=31,
    author=AUTHOR,
    treated=TREATED,
    control=CONTROL,
    metric=METRIC,
    window="all_pre_vs_post",
    ci_low=fe_ci_low,
    ci_high=fe_ci_high,
    notes="Two-way FE DiD using US + Canada only. Coefficient on digital_feature_available after absorbing market and week fixed effects; OLS with Newey-West/HAC errors via sd.smf_ols(maxlags=8).",
)

print(f"Two-way FE DiD ATT: {fe_att:+.4f} RPS")
print(f"95% CI: [{fe_ci_low:+.4f}, {fe_ci_high:+.4f}]")
print(f"p-value: {fe_p:.3e}")

Two-way FE DiD ATT: +0.2459 RPS
95% CI: [+0.1879, +0.3039]
p-value: 9.371e-17


## Four-way comparison

In [6]:
prepost = us_ca.groupby(["market", "post_launch"])[METRIC].mean().unstack()
prepost.columns = ["pre", "post"]
prepost["change"] = prepost["post"] - prepost["pre"]

naive = float(prepost.loc[TREATED, "change"])
canada_change = float(prepost.loc[CONTROL, "change"])
manual_did = naive - canada_change

comparison = pd.DataFrame([
    {"method":"naive_before_after", "estimate":naive, "ci_low":None, "ci_high":None, "p_value":None},
    {"method":"manual_2x2_did", "estimate":manual_did, "ci_low":None, "ci_high":None, "p_value":None},
    {"method":"simple_regression_did", "estimate":simple_att, "ci_low":simple_ci_low, "ci_high":simple_ci_high, "p_value":simple_p},
    {"method":"two_way_fe_did", "estimate":fe_att, "ci_low":fe_ci_low, "ci_high":fe_ci_high, "p_value":fe_p},
])
comparison

                  method  ...                                              notes
0     naive_before_after  ...           Descriptive U.S. before/after lift only.
1         manual_2x2_did  ...  Manual 2x2 DiD using Canada; no standard error...
2  simple_regression_did  ...  Coefficient on pilot_market:post_launch from O...
3         two_way_fe_did  ...  Coefficient on digital_feature_available from ...

[4 rows x 10 columns]

## Plain-English readout

- Naive before/after: **+0.3421 RPS**. This is descriptive only.
- Manual 2×2 DiD: **+0.2459 RPS** after subtracting Canada’s baseline movement.
- Simple regression DiD: **+0.2459 RPS**, 95% CI **[-0.0203, +0.5122]**, p = **0.0703**. With Newey-West/HAC errors, this CI includes zero, so the point estimate is positive but not statistically significant at the 5% threshold.
- Two-way FE DiD: **+0.2459 RPS**, 95% CI **[+0.1879, +0.3039]**, p < **0.001**. This estimate remains positive after absorbing market and week fixed effects.

The point estimate is consistent across manual DiD, simple regression DiD, and two-way FE DiD. The main change from Milestone #3 is that Milestone #4 adds uncertainty through standard errors, confidence intervals, and p-values.